# 06 - Text + structured features

Adds the listing text to the structured columns used in notebooks 04 and 05, in two ways:
1. one Ridge on the TF-IDF words and the structured columns together (compare with Ridge in notebook 04);
2. LightGBM on the structured columns plus one extra column: the price predicted from the text alone (compare with LightGBM in notebook 05).

The `text` column was built in notebook 03 (title + description, with price mentions and numbers removed).

In [1]:
import os

import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from scipy.sparse import hstack, vstack
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import (mean_absolute_error, mean_absolute_percentage_error,
                             r2_score, root_mean_squared_error)
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler

NUMERIC = ["area", "floor_count", "frontage_width", "house_depth", "road_width",
           "bedroom_count", "bathroom_count"]
CATEGORICAL = ["property_type_name", "province_name", "district_name", "ward_name",
               "street_name", "project_name", "house_direction", "balcony_direction"]

## Data

In [2]:
df = pd.read_parquet("../data/processed/model_input.parquet",
                     columns=["text", "price", "split"] + NUMERIC + CATEGORICAL)

train = df[df["split"] == "train"]
val = df[df["split"] == "val"]
test = df[df["split"] == "test"]

log_y_train = np.log1p(train["price"])
log_y_val = np.log1p(val["price"])

In [3]:
def evaluate(y_true, y_pred):
    return {
        "MAE (bn VND)": mean_absolute_error(y_true, y_pred) / 1e9,
        "RMSE (bn VND)": root_mean_squared_error(y_true, y_pred) / 1e9,
        "R2": r2_score(y_true, y_pred),
        "MAPE (%)": 100 * mean_absolute_percentage_error(y_true, y_pred),
    }


results = {}

## TF-IDF

Each listing becomes a sparse vector of word weights. The vocabulary is learned from the training set only:
words that appear in fewer than 5 listings are dropped and the 50,000 most frequent words are kept.

In [4]:
vectorizer = TfidfVectorizer(min_df=5, max_features=50_000, sublinear_tf=True, dtype=np.float32)

X_train = vectorizer.fit_transform(train["text"])
X_val = vectorizer.transform(val["text"])
X_test = vectorizer.transform(test["text"])
X_train.shape

(1677900, 23233)

## 1. Ridge on text + structured columns

The structured columns get the same preprocessing as Ridge in notebook 04 (log, fill missing, scale, one-hot), then they are placed next to the TF-IDF columns.

In [5]:
numeric = make_pipeline(
    FunctionTransformer(np.log1p),
    SimpleImputer(strategy="median", add_indicator=True),
    StandardScaler(),
)
categorical = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50)
preprocess = ColumnTransformer([("num", numeric, NUMERIC), ("cat", categorical, CATEGORICAL)])

# structured columns: learned on train only
S_train = preprocess.fit_transform(train)
S_val = preprocess.transform(val)
S_test = preprocess.transform(test)

# TF-IDF columns + structured columns, side by side
both_train = hstack([X_train, S_train], format="csr", dtype=np.float32)
both_val = hstack([X_val, S_val], format="csr", dtype=np.float32)
both_test = hstack([X_test, S_test], format="csr", dtype=np.float32)
both_train.shape

(1677900, 29592)

Each `alpha` is trained on the training set and scored on the validation set. The best one is refitted on train + validation.

In [6]:
rows, best_mae = [], np.inf
for alpha in [1, 10, 100]:
    model = Ridge(alpha=alpha).fit(both_train, log_y_train)
    mae = mean_absolute_error(val["price"], np.expm1(model.predict(both_val))) / 1e9
    rows.append({"alpha": alpha, "val MAE (bn VND)": mae})

    if mae < best_mae:
        best_mae, best_alpha, best_ridge = mae, alpha, model

print("Best alpha:", best_alpha)
pd.DataFrame(rows)

Best alpha: 1


,alpha,val MAE (bn VND)
0,1,4.519394
1,10,4.545956
2,100,4.696974


In [7]:
final_ridge = Ridge(alpha=best_alpha).fit(vstack([both_train, both_val]), pd.concat([log_y_train, log_y_val]))

results["Ridge: text + structured"] = evaluate(test["price"], np.expm1(final_ridge.predict(both_test)))
results["Ridge: text + structured"]

{'MAE (bn VND)': 4.9350906657952915,
 'RMSE (bn VND)': 21.60646365068447,
 'R2': 0.6319373953795147,
 'MAPE (%)': 28.08969421852714}

### Which words matter?

The first 50,000 coefficients belong to the words. Positive words push the predicted price up, negative words push it down.

In [8]:
words = vectorizer.get_feature_names_out()
coefficients = pd.Series(final_ridge.coef_[:len(words)], index=words).sort_values()

pd.DataFrame({
    "cheaper": coefficients.head(15).index,
    "more expensive": coefficients.tail(15).index[::-1],
})

,cheaper,more expensive
0,rsx,batdongsanchienluoc
1,serepok,costamigo
2,binhbds,amadive
3,mokuzo,saroma
4,kbang,elyse
5,bhk,holm
6,pil,sungrup
7,vshape,flycam
8,ruộng,nltc
9,lua,anara


## 2. LightGBM + text prediction

A Ridge trained on the text alone predicts a (log) price for every listing, and LightGBM uses it as one more column next to the structured ones.

For the training rows this prediction must come from a model that has not seen those rows, otherwise it would be too accurate and LightGBM would trust it too much.
`cross_val_predict` does this: it splits the training set into 5 parts and predicts each part with a Ridge trained on the other 4.
The text Ridge reuses the `alpha` chosen above.

In [9]:
text_ridge = Ridge(alpha=best_alpha)
train_text_pred = cross_val_predict(text_ridge, X_train, log_y_train, cv=5)
text_ridge.fit(X_train, log_y_train)

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1).fit(train[CATEGORICAL])


def lgb_features(part, text_pred):
    X = part[NUMERIC + CATEGORICAL].copy()
    X[CATEGORICAL] = encoder.transform(part[CATEGORICAL]).astype(int)
    X["text_pred"] = text_pred
    return X


L_train = lgb_features(train, train_text_pred)
L_val = lgb_features(val, text_ridge.predict(X_val))
L_test = lgb_features(test, text_ridge.predict(X_test))

In [10]:
lgbm = LGBMRegressor(n_estimators=2000, learning_rate=0.05, num_leaves=63, random_state=42, verbose=-1)
lgbm.fit(
    L_train, log_y_train,
    eval_set=[(L_val, log_y_val)],
    eval_metric="mae",
    categorical_feature=CATEGORICAL,
    callbacks=[early_stopping(50), log_evaluation(100)],
)

results["LightGBM + text prediction"] = evaluate(test["price"], np.expm1(lgbm.predict(L_test)))
results["LightGBM + text prediction"]

Training until validation scores don't improve for 50 rounds
[100]	valid_0's l1: 0.254938	valid_0's l2: 0.132049
[200]	valid_0's l1: 0.23406	valid_0's l2: 0.114739
[300]	valid_0's l1: 0.226809	valid_0's l2: 0.109595
[400]	valid_0's l1: 0.223139	valid_0's l2: 0.107146
[500]	valid_0's l1: 0.220979	valid_0's l2: 0.105717
[600]	valid_0's l1: 0.219569	valid_0's l2: 0.104816
[700]	valid_0's l1: 0.218615	valid_0's l2: 0.104243
[800]	valid_0's l1: 0.217716	valid_0's l2: 0.103698
[900]	valid_0's l1: 0.217045	valid_0's l2: 0.103257
[1000]	valid_0's l1: 0.216493	valid_0's l2: 0.102872
[1100]	valid_0's l1: 0.215995	valid_0's l2: 0.102555
[1200]	valid_0's l1: 0.215655	valid_0's l2: 0.102319
[1300]	valid_0's l1: 0.215249	valid_0's l2: 0.102046
[1400]	valid_0's l1: 0.214957	valid_0's l2: 0.101859
[1500]	valid_0's l1: 0.214621	valid_0's l2: 0.101657
[1600]	valid_0's l1: 0.214348	valid_0's l2: 0.101476
[1700]	valid_0's l1: 0.214186	valid_0's l2: 0.10138
[1800]	valid_0's l1: 0.213971	valid_0's l2: 0.101

{'MAE (bn VND)': 4.223372342320444,
 'RMSE (bn VND)': 17.316701809435738,
 'R2': 0.7635797514773796,
 'MAPE (%)': 25.049592693364325}

## Results

In [11]:
table = pd.DataFrame(results).T

os.makedirs("../reports/metrics", exist_ok=True)
table.to_csv("../reports/metrics/06_text_tfidf.csv")

table.style.format("{:,.3f}")

,MAE (bn VND),RMSE (bn VND),R2,MAPE (%)
Ridge: text + structured,4.935,21.606,0.632,28.090
LightGBM + text prediction,4.223,17.317,0.764,25.050


## Save predictions

The (log) predictions on validation and test are saved for the ensemble in notebook 10.
The validation predictions of the Ridge come from the model trained on train only (`best_ridge`), because `final_ridge` has already seen the validation rows.

In [12]:
predictions = pd.concat([
    pd.DataFrame({"split": "val",
                  "Ridge text + structured": best_ridge.predict(both_val),
                  "LightGBM + text": lgbm.predict(L_val)}, index=val.index),
    pd.DataFrame({"split": "test",
                  "Ridge text + structured": final_ridge.predict(both_test),
                  "LightGBM + text": lgbm.predict(L_test)}, index=test.index),
])

os.makedirs("../reports/predictions", exist_ok=True)
predictions.to_parquet("../reports/predictions/06_text_tfidf.parquet")
predictions.head()

,split,Ridge text + structured,LightGBM + text
1677900,val,22.887657,22.326154
1677901,val,22.249714,22.238223
1677902,val,21.394213,21.701439
1677903,val,24.135427,23.860683
1677904,val,21.911989,21.924883
